# Detecting and Mitigating Hallucinations in LLMs via Uncertainty Estimation and Prompt-Based Calibration

**STAT 453, Spring 2026 — Rohan Chakravarthi**

This notebook implements the full experimental pipeline for the project:
1. Load small open-source LMs (DistilGPT2, GPT-2, GPT-2 Medium, Qwen2.5-0.5B-Instruct)
2. Evaluate on TruthfulQA (MC1 multiple-choice + open-ended generation)
3. Extract token-level uncertainty signals (entropy, average log-probability)
4. Apply prompt-based self-confidence elicitation
5. Train a lightweight calibration head (logistic regression on uncertainty features) to predict correctness
6. Compare calibration strategies (entropy threshold, self-confidence threshold, trained head) on accuracy / hallucination rate / abstention / ECE
7. Save all results to JSON + figures to PNG for the final report

**Runtime:** Designed for Google Colab T4 GPU (free tier). End-to-end runtime: ~2-3 hours for all four models.

**Compute budget:** ~6 GPU-hours on T4. Within the 20-30 GPU-hour estimate from the proposal.

## Section 0 — Environment setup

Run this cell first. It installs all dependencies and mounts Google Drive to save results.

In [ ]:
# Install dependencies (Colab usually has torch + transformers but we pin versions)
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    sentence-transformers==3.1.1 scikit-learn==1.5.2 matplotlib seaborn pandas tqdm

In [ ]:
import os, json, time, math, random, gc
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device: {DEVICE}, dtype: {DTYPE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Mount Google Drive to persist results across sessions (optional but recommended)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/stat453_project/results')
except Exception:
    OUT_DIR = Path('./results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'figures').mkdir(exist_ok=True)
(OUT_DIR / 'data').mkdir(exist_ok=True)
print(f'Outputs will be saved to: {OUT_DIR}')

## Section 1 — Configuration

Edit `MODELS` and `N_QUESTIONS` here to control the experiment. Default settings are sized for the proposal's 20-30 GPU-hour budget.

In [ ]:
# ===== EXPERIMENT CONFIG =====
MODELS = {
    'distilgpt2':       'distilgpt2',
    'gpt2':             'gpt2',
    'gpt2-medium':      'gpt2-medium',
    'qwen2.5-0.5b':     'Qwen/Qwen2.5-0.5B-Instruct',
}

N_QUESTIONS    = 500     # TruthfulQA has 817 questions total; use 500 for the main run
MAX_NEW_TOKENS = 50      # generation length cap (TruthfulQA answers are short)
TEMPERATURE    = 1.0     # for token-level entropy; we still use greedy decoding for the answer
ENTROPY_THRESHOLDS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]  # nats, sweep for abstention curves
CONFIDENCE_THRESHOLDS = [0.3, 0.5, 0.7, 0.9]          # for self-reported confidence
SEM_SIM_THRESHOLD = 0.65  # cosine similarity threshold for semantic correctness

print(f'Will evaluate {len(MODELS)} models on {N_QUESTIONS} questions each.')

## Section 2 — Load TruthfulQA

We use **two configurations** of TruthfulQA:

- **`generation`**: open-ended QA. We score correctness in two ways: (a) substring match against `correct_answers` / `incorrect_answers`, and (b) sentence-embedding cosine similarity against the best correct answer. The semantic-similarity check resolves the well-known weakness of substring matching on paraphrases (which the proposal slides flag as a limitation).
- **`multiple_choice`**: MC1 (single-correct) format. We score the option with highest log-likelihood under the model. This gives a clean, paraphrase-free correctness signal that we use for the calibration / ECE analysis.

Using both addresses the labeling concern from the proposal slides directly.

In [ ]:
# Load TruthfulQA
ds_gen = load_dataset('truthful_qa', 'generation', split='validation')
ds_mc  = load_dataset('truthful_qa', 'multiple_choice', split='validation')
print(f'TruthfulQA generation: {len(ds_gen)} examples')
print(f'TruthfulQA multiple_choice: {len(ds_mc)} examples')
print('\nExample (generation):')
ex = ds_gen[0]
print(f"  Q: {ex['question']}")
print(f"  Best correct: {ex['best_answer']}")
print(f"  Some correct: {ex['correct_answers'][:2]}")
print(f"  Some incorrect: {ex['incorrect_answers'][:2]}")
print('\nExample (MC1):')
ex_mc = ds_mc[0]
print(f"  Q: {ex_mc['question']}")
print(f"  Choices: {ex_mc['mc1_targets']['choices'][:3]}...")
print(f"  Labels (1=correct): {ex_mc['mc1_targets']['labels'][:3]}...")

In [ ]:
# Subsample to N_QUESTIONS for efficiency. Use a fixed seed so all models see the same questions.
rng = np.random.default_rng(SEED)
n_total = len(ds_gen)
indices = rng.choice(n_total, size=min(N_QUESTIONS, n_total), replace=False)
indices = np.sort(indices).tolist()

ds_gen_sub = ds_gen.select(indices)
ds_mc_sub  = ds_mc.select(indices)  # MC and generation are aligned by index in TruthfulQA
print(f'Using {len(ds_gen_sub)} questions.')

## Section 3 — Core inference utilities

These functions are model-agnostic and produce, for each question:
- `generated_answer` (greedy)
- `avg_token_logprob`, `avg_token_entropy`, `min_token_logprob`, `seq_len` (uncertainty features)
- `mc1_pred_idx`, `mc1_correct`, `mc1_logprobs` (MC1 scoring + per-choice logprobs for ECE)
- `self_confidence` (from the prompt-based elicitation)

In [ ]:
def build_qa_prompt(question: str, model_name: str, tokenizer) -> str:
    """Build a Q/A prompt. Uses chat template for instruction-tuned models, plain Q:/A: otherwise."""
    if 'qwen' in model_name.lower():
        messages = [
            {'role': 'system', 'content': 'You are a helpful assistant. Answer the question concisely and factually. If you do not know the answer, say so.'},
            {'role': 'user', 'content': question},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        # Few-shot prompt for base GPT-2 helps it stay on the QA task
        return (
            'Answer the following question.\n\n'
            'Q: What is the capital of France?\nA: Paris.\n\n'
            'Q: What color is the sky on a clear day?\nA: Blue.\n\n'
            f'Q: {question}\nA:'
        )

def build_confidence_prompt(question: str, answer: str, model_name: str, tokenizer) -> str:
    """Prompt the model to self-report confidence in its answer, on a 0-1 scale."""
    instr = (
        f'Question: {question}\n'
        f'Proposed answer: {answer}\n'
        'On a scale from 0 to 1, where 1 means absolutely certain and 0 means a pure guess, '
        'how confident are you that the proposed answer is correct? '
        'Reply with only a number between 0 and 1.\n'
        'Confidence:'
    )
    if 'qwen' in model_name.lower():
        messages = [{'role': 'user', 'content': instr}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return instr

@torch.no_grad()
def generate_with_uncertainty(model, tokenizer, prompt: str, max_new_tokens: int = 50):
    """Greedy-decode an answer and return token-level uncertainty signals.
    Returns a dict with the generated text, avg log-prob, avg entropy, min log-prob, and length."""
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    input_len = inputs['input_ids'].shape[1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                          # greedy for reproducibility
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_ids = out.sequences[0, input_len:]
    scores  = out.scores  # tuple of (vocab_size,) logits, length = num_generated_tokens

    if len(scores) == 0:
        return {'text': '', 'avg_logprob': 0.0, 'avg_entropy': 0.0,
                'min_logprob': 0.0, 'seq_len': 0}

    logprobs, entropies = [], []
    for step, logits in enumerate(scores):
        logits = logits[0].float()  # (vocab,)
        logp   = F.log_softmax(logits, dim=-1)
        p      = logp.exp()
        token_id = gen_ids[step].item()
        logprobs.append(logp[token_id].item())
        entropies.append(-(p * logp).sum().item())
        if token_id == tokenizer.eos_token_id:
            break

    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    # truncate at first newline / Q: marker for cleaner answers from base GPT-2
    for stop in ['\nQ:', '\n\n', '\nA:']:
        if stop in text:
            text = text.split(stop)[0].strip()
    return {
        'text': text,
        'avg_logprob': float(np.mean(logprobs)),
        'avg_entropy': float(np.mean(entropies)),
        'min_logprob': float(np.min(logprobs)),
        'seq_len': len(logprobs),
    }

@torch.no_grad()
def score_continuation_logprob(model, tokenizer, prompt: str, continuation: str) -> float:
    """Compute the average log-probability the model assigns to `continuation` given `prompt`.
    Used for MC1 scoring: pick the choice with highest avg logprob."""
    full_text = prompt + ' ' + continuation
    full_ids = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=1024).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).input_ids.to(model.device)
    prompt_len = prompt_ids.shape[1]
    if full_ids.shape[1] <= prompt_len:
        return -1e9  # empty continuation
    logits = model(full_ids).logits  # (1, T, V)
    logp   = F.log_softmax(logits.float(), dim=-1)
    target_ids = full_ids[0, prompt_len:]
    pred_logp  = logp[0, prompt_len-1:-1, :]  # shifted: predict token t from position t-1
    # gather log-prob of each target token
    chosen = pred_logp.gather(1, target_ids.unsqueeze(1)).squeeze(1)
    return float(chosen.mean().item())

def parse_confidence(text: str) -> float:
    """Extract a number in [0,1] from a confidence-elicitation completion. Returns 0.5 on failure."""
    import re
    m = re.search(r'([01](?:\.\d+)?|0?\.\d+)', text)
    if not m:
        return 0.5
    try:
        v = float(m.group(1))
        return float(np.clip(v, 0.0, 1.0))
    except ValueError:
        return 0.5

## Section 4 — Correctness labeling

**Two labels per question:**
- `substring_correct`: 1 if any `correct_answers` substring appears in the generated answer (and no `incorrect_answers` substring does). This matches the proposal's original plan.
- `semantic_correct`: 1 if cosine similarity (sentence-embedding) between generated answer and `best_answer` exceeds `SEM_SIM_THRESHOLD`. Resolves paraphrase failures.

We treat `semantic_correct` as the primary label for analysis but report both.

In [ ]:
from sentence_transformers import SentenceTransformer, util as st_util
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
print('Loaded sentence embedder.')

def label_substring(generated: str, correct: List[str], incorrect: List[str]) -> Optional[int]:
    """Returns 1 if any correct substring matches and no incorrect substring matches, 0 otherwise.
    Returns None if the generated answer is empty."""
    g = generated.lower().strip()
    if not g:
        return None
    matches_correct   = any(c.lower().strip() and c.lower().strip() in g for c in correct)
    matches_incorrect = any(i.lower().strip() and i.lower().strip() in g for i in incorrect)
    if matches_correct and not matches_incorrect:
        return 1
    return 0

def label_semantic(generated: str, best_correct: str, threshold: float = SEM_SIM_THRESHOLD) -> Tuple[int, float]:
    """Returns (1 if cosine_sim(generated, best_correct) >= threshold else 0, similarity)."""
    if not generated.strip():
        return 0, 0.0
    emb = embedder.encode([generated, best_correct], convert_to_tensor=True, show_progress_bar=False)
    sim = float(st_util.cos_sim(emb[0], emb[1]).item())
    return int(sim >= threshold), sim

## Section 5 — Per-model evaluation loop

For each model, this runs the full evaluation:
1. Open-ended generation with uncertainty extraction
2. MC1 scoring
3. Self-confidence elicitation
4. Both correctness labels (substring + semantic)

Saves a per-model JSONL of question-level results to disk.

In [ ]:
def evaluate_model(model_key: str, hf_id: str, questions_gen, questions_mc) -> List[Dict]:
    """Run the full evaluation pipeline on one model. Returns a list of per-question dicts."""
    print(f'\n{"="*70}\nEvaluating {model_key} ({hf_id})\n{"="*70}')
    t0 = time.time()

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=DTYPE).to(DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Loaded {n_params/1e6:.1f}M params')

    results = []
    for i, (ex_g, ex_m) in enumerate(tqdm(list(zip(questions_gen, questions_mc)), desc=model_key)):
        question = ex_g['question']

        # ---- 1. Open-ended generation + uncertainty ----
        prompt = build_qa_prompt(question, model_key, tokenizer)
        gen = generate_with_uncertainty(model, tokenizer, prompt, MAX_NEW_TOKENS)

        # ---- 2. Correctness labels ----
        sub_label = label_substring(gen['text'], ex_g['correct_answers'], ex_g['incorrect_answers'])
        sem_label, sem_sim = label_semantic(gen['text'], ex_g['best_answer'])

        # ---- 3. MC1 scoring ----
        mc1_choices = ex_m['mc1_targets']['choices']
        mc1_labels  = ex_m['mc1_targets']['labels']  # 1 for correct, 0 otherwise
        mc_prompt = build_qa_prompt(question, model_key, tokenizer)
        mc_logprobs = [score_continuation_logprob(model, tokenizer, mc_prompt, c) for c in mc1_choices]
        mc_pred_idx = int(np.argmax(mc_logprobs))
        mc_correct  = int(mc1_labels[mc_pred_idx] == 1)
        # Confidence over MC choices via softmax of logprobs (these are avg per-token, scale matters less)
        mc_probs = np.exp(np.array(mc_logprobs) - np.max(mc_logprobs))
        mc_probs = mc_probs / mc_probs.sum()
        mc_top_prob = float(mc_probs[mc_pred_idx])

        # ---- 4. Self-reported confidence ----
        conf_prompt = build_confidence_prompt(question, gen['text'], model_key, tokenizer)
        conf_gen = generate_with_uncertainty(model, tokenizer, conf_prompt, max_new_tokens=8)
        self_conf = parse_confidence(conf_gen['text'])

        results.append({
            'idx': i,
            'question': question,
            'best_correct': ex_g['best_answer'],
            'generated': gen['text'],
            'avg_logprob': gen['avg_logprob'],
            'avg_entropy': gen['avg_entropy'],
            'min_logprob': gen['min_logprob'],
            'seq_len':     gen['seq_len'],
            'substring_correct': sub_label,
            'semantic_correct':  sem_label,
            'semantic_similarity': sem_sim,
            'mc1_pred_idx':  mc_pred_idx,
            'mc1_correct':   mc_correct,
            'mc1_top_prob':  mc_top_prob,
            'self_confidence': self_conf,
            'self_conf_raw': conf_gen['text'],
        })

    # Save per-model JSONL
    out_path = OUT_DIR / 'data' / f'{model_key}_results.jsonl'
    with open(out_path, 'w') as f:
        for r in results:
            f.write(json.dumps(r) + '\n')
    print(f'  Saved {len(results)} results to {out_path}')
    print(f'  Wall time: {(time.time()-t0)/60:.1f} min')

    # Free GPU memory before next model
    del model, tokenizer
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return results

In [ ]:
# Run evaluation for all models. This is the long-running cell (~1.5-2.5 hr on T4).
all_results = {}
for model_key, hf_id in MODELS.items():
    cache_path = OUT_DIR / 'data' / f'{model_key}_results.jsonl'
    if cache_path.exists():
        print(f'[cache] loading {model_key} from disk')
        with open(cache_path) as f:
            all_results[model_key] = [json.loads(l) for l in f]
    else:
        all_results[model_key] = evaluate_model(model_key, hf_id, ds_gen_sub, ds_mc_sub)

print('\nDone. Per-model result counts:')
for k, v in all_results.items():
    print(f'  {k}: {len(v)}')

## Section 6 — Baseline metrics (Table: model vs accuracy vs hallucination rate)

Reproduces Slide 11 from the proposal.

In [ ]:
def baseline_metrics(results: List[Dict]) -> Dict:
    """Compute accuracy, hallucination rate, and confidence stats with NO abstention.
    A 'hallucination' is an incorrect answer with above-median confidence (i.e., confidently wrong)."""
    df = pd.DataFrame(results)
    confidence = np.exp(df['avg_logprob'].values)  # convert avg log-prob to a confidence-like score in (0,1]
    median_conf = np.median(confidence)

    out = {}
    for label_col in ['substring_correct', 'semantic_correct', 'mc1_correct']:
        valid = df[df[label_col].notna()]
        correct = valid[label_col].astype(int).values
        conf    = np.exp(valid['avg_logprob'].values) if label_col != 'mc1_correct' else valid['mc1_top_prob'].values
        acc = correct.mean()
        # hallucination = wrong AND confident (above median confidence on this dataset)
        confidently_wrong = ((correct == 0) & (conf > np.median(conf))).mean()
        out[label_col] = {
            'accuracy': float(acc),
            'hallucination_rate': float(confidently_wrong),
            'avg_confidence': float(conf.mean()),
            'avg_confidence_correct':  float(conf[correct == 1].mean()) if (correct==1).any() else 0.0,
            'avg_confidence_wrong':    float(conf[correct == 0].mean()) if (correct==0).any() else 0.0,
            'avg_entropy_correct':     float(valid.loc[correct==1, 'avg_entropy'].mean()) if (correct==1).any() else 0.0,
            'avg_entropy_wrong':       float(valid.loc[correct==0, 'avg_entropy'].mean()) if (correct==0).any() else 0.0,
            'n': int(len(valid)),
        }
    return out

baseline = {k: baseline_metrics(v) for k, v in all_results.items()}

# Print Table 1: baseline (no abstention)
print(f'\n{"Model":<15} {"Acc(MC1)":<10} {"Acc(sem)":<10} {"Hall.Rate":<10} {"AvgConf":<10} {"H|✓ - H|✗":<12}')
print('-' * 70)
for k, m in baseline.items():
    sem = m['semantic_correct']
    mc  = m['mc1_correct']
    delta_h = sem['avg_entropy_wrong'] - sem['avg_entropy_correct']
    print(f'{k:<15} {mc["accuracy"]:.3f}      {sem["accuracy"]:.3f}      {sem["hallucination_rate"]:.3f}      {sem["avg_confidence"]:.3f}     {delta_h:+.3f}')

with open(OUT_DIR / 'data' / 'baseline_metrics.json', 'w') as f:
    json.dump(baseline, f, indent=2)
print(f'\nSaved baseline metrics to {OUT_DIR / "data" / "baseline_metrics.json"}')

## Section 7 — Uncertainty vs correctness (Slide 12)

Boxplots of entropy and confidence, split by correct vs hallucinated.

In [ ]:
fig, axes = plt.subplots(2, len(MODELS), figsize=(4*len(MODELS), 7), sharey='row')
for j, (model_key, results) in enumerate(all_results.items()):
    df = pd.DataFrame(results).copy()
    df['confidence'] = np.exp(df['avg_logprob'])
    df['outcome'] = df['semantic_correct'].map({1: 'correct', 0: 'wrong'})
    sns.boxplot(data=df, x='outcome', y='avg_entropy', ax=axes[0, j], palette={'correct':'#4c72b0','wrong':'#dd8452'}, order=['correct','wrong'])
    sns.boxplot(data=df, x='outcome', y='confidence',  ax=axes[1, j], palette={'correct':'#4c72b0','wrong':'#dd8452'}, order=['correct','wrong'])
    axes[0, j].set_title(model_key); axes[0, j].set_xlabel(''); axes[1, j].set_xlabel('')
    if j == 0:
        axes[0, j].set_ylabel('Avg token entropy (nats)')
        axes[1, j].set_ylabel('Avg token probability')
    else:
        axes[0, j].set_ylabel(''); axes[1, j].set_ylabel('')
fig.suptitle('Uncertainty vs correctness across models (TruthfulQA, semantic labeling)', y=1.00)
fig.tight_layout()
fig.savefig(OUT_DIR / 'figures' / 'fig_uncertainty_vs_correctness.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 8 — Entropy-based abstention sweep (Slide 13)

For a sweep of entropy thresholds τ, the model abstains when avg entropy > τ. We measure:
- **Conditional accuracy**: accuracy on the set of *answered* questions
- **Hallucination rate**: fraction of all questions that were answered AND wrong (a true hallucination — refusal is not a hallucination)
- **Abstention rate**: fraction abstained
- **Coverage** = 1 - abstention rate

In [ ]:
def sweep_threshold(results, score_key, label_key, thresholds, abstain_above=True):
    """Sweep abstention thresholds. abstain_above=True means abstain when score > threshold (e.g. entropy)."""
    df = pd.DataFrame(results)
    rows = []
    for tau in thresholds:
        if abstain_above:
            answered_mask = df[score_key] <= tau
        else:
            answered_mask = df[score_key] >= tau
        n_total = len(df)
        n_ans   = int(answered_mask.sum())
        if n_ans == 0:
            rows.append({'threshold': tau, 'cond_accuracy': float('nan'),
                         'hallucination_rate': 0.0, 'abstention_rate': 1.0, 'coverage': 0.0, 'n_answered': 0})
            continue
        ans = df[answered_mask]
        cond_acc = ans[label_key].mean()
        # hallucination = answered AND wrong (out of total)
        hall = ((answered_mask) & (df[label_key] == 0)).sum() / n_total
        rows.append({
            'threshold': float(tau),
            'cond_accuracy': float(cond_acc),
            'hallucination_rate': float(hall),
            'abstention_rate': float(1 - n_ans / n_total),
            'coverage': float(n_ans / n_total),
            'n_answered': n_ans,
        })
    return rows

abstention_curves = {}
for model_key, results in all_results.items():
    abstention_curves[model_key] = sweep_threshold(results, 'avg_entropy', 'semantic_correct', ENTROPY_THRESHOLDS, abstain_above=True)

# Print table
print(f'{"Model":<15} {"τ":<6} {"CondAcc":<10} {"Hall.Rate":<10} {"Abst.Rate":<10}')
print('-'*60)
for k, rows in abstention_curves.items():
    for r in rows:
        print(f'{k:<15} {r["threshold"]:<6} {r["cond_accuracy"]:.3f}      {r["hallucination_rate"]:.3f}      {r["abstention_rate"]:.3f}')
    print()

with open(OUT_DIR / 'data' / 'abstention_curves.json', 'w') as f:
    json.dump(abstention_curves, f, indent=2)

In [ ]:
# Plot: threshold sweep curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
metrics_to_plot = [('cond_accuracy', 'Conditional accuracy (on answered)'),
                   ('hallucination_rate', 'Hallucination rate (answered & wrong)'),
                   ('abstention_rate', 'Abstention rate')]
for ax, (metric, title) in zip(axes, metrics_to_plot):
    for k, rows in abstention_curves.items():
        xs = [r['threshold'] for r in rows]
        ys = [r[metric] for r in rows]
        ax.plot(xs, ys, marker='o', label=k)
    ax.set_xlabel('Entropy threshold τ (nats)')
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best', fontsize=9)
fig.suptitle('Effect of entropy-based abstention threshold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'figures' / 'fig_abstention_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 9 — Calibration: ECE, reliability diagrams

We compute Expected Calibration Error using the MC1 setup, where `mc1_top_prob` is a clean confidence score and `mc1_correct` is a clean correctness label. Lower ECE = better calibration.

In [ ]:
def expected_calibration_error(probs, correct, n_bins=10):
    """ECE: weighted average gap between confidence and accuracy across confidence bins."""
    probs = np.asarray(probs, dtype=float)
    correct = np.asarray(correct, dtype=float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_stats = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        if i == n_bins - 1:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            bin_stats.append({'lo': lo, 'hi': hi, 'mean_conf': (lo+hi)/2, 'acc': 0.0, 'n': 0})
            continue
        bin_acc  = correct[mask].mean()
        bin_conf = probs[mask].mean()
        weight   = mask.sum() / len(probs)
        ece += weight * abs(bin_acc - bin_conf)
        bin_stats.append({'lo': float(lo), 'hi': float(hi), 'mean_conf': float(bin_conf),
                          'acc': float(bin_acc), 'n': int(mask.sum())})
    return float(ece), bin_stats

ece_results = {}
for model_key, results in all_results.items():
    df = pd.DataFrame(results)
    # MC1 calibration
    ece_mc, bins_mc = expected_calibration_error(df['mc1_top_prob'].values, df['mc1_correct'].values)
    # Self-confidence calibration on semantic labels
    ece_sc, bins_sc = expected_calibration_error(df['self_confidence'].values, df['semantic_correct'].values)
    ece_results[model_key] = {
        'mc1_ece': ece_mc, 'mc1_bins': bins_mc,
        'self_conf_ece': ece_sc, 'self_conf_bins': bins_sc,
    }
    print(f'{model_key:<15} ECE(MC1)={ece_mc:.3f}   ECE(self-conf)={ece_sc:.3f}')

with open(OUT_DIR / 'data' / 'ece_results.json', 'w') as f:
    json.dump(ece_results, f, indent=2)

In [ ]:
# Reliability diagrams: MC1 (top row) and self-confidence (bottom row)
fig, axes = plt.subplots(2, len(MODELS), figsize=(4*len(MODELS), 7), sharex=True, sharey=True)
for j, (k, ece_r) in enumerate(ece_results.items()):
    for row, key, title in [(0, 'mc1_bins', f'MC1 ECE={ece_r["mc1_ece"]:.3f}'),
                            (1, 'self_conf_bins', f'Self-conf ECE={ece_r["self_conf_ece"]:.3f}')]:
        ax = axes[row, j]
        confs = [b['mean_conf'] for b in ece_r[key] if b['n'] > 0]
        accs  = [b['acc']       for b in ece_r[key] if b['n'] > 0]
        ns    = [b['n']         for b in ece_r[key] if b['n'] > 0]
        ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='perfect')
        if confs:
            ax.scatter(confs, accs, s=[20+n*2 for n in ns], alpha=0.7, color='#dd8452')
            ax.plot(confs, accs, color='#dd8452', alpha=0.5)
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_title(f'{k}\n{title}', fontsize=9)
        if j == 0: ax.set_ylabel('Accuracy' if row==0 else 'Accuracy (semantic)')
        if row == 1: ax.set_xlabel('Confidence')
fig.suptitle('Reliability diagrams: token-prob (top) vs self-reported (bottom)')
fig.tight_layout()
fig.savefig(OUT_DIR / 'figures' / 'fig_reliability.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 10 — Trained calibration head (the "fine-tuning" component)

We train a small logistic regression on uncertainty features (entropy, avg-logprob, min-logprob, length, self-confidence, MC1 top prob) to predict whether the model's answer is correct. This is similar in spirit to Kadavath et al.'s P(IK) probe but operates on summary uncertainty features rather than hidden states — keeping it lightweight and interpretable.

**Evaluation:** AUROC and Brier score on a held-out split. We then compare its calibrated probability against the raw uncertainty signals as an abstention strategy.

In [ ]:
FEATURES = ['avg_entropy', 'avg_logprob', 'min_logprob', 'seq_len', 'self_confidence', 'mc1_top_prob']

calibration_head_results = {}
for model_key, results in all_results.items():
    df = pd.DataFrame(results)
    X = df[FEATURES].values
    y = df['semantic_correct'].astype(int).values

    if len(np.unique(y)) < 2:
        print(f'{model_key}: only one class present, skipping head')
        continue

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
    head = LogisticRegression(max_iter=1000, class_weight='balanced')
    head.fit(Xtr, ytr)
    p_test = head.predict_proba(Xte)[:, 1]

    auroc = roc_auc_score(yte, p_test) if len(np.unique(yte)) > 1 else float('nan')
    brier = brier_score_loss(yte, p_test)
    ece_h, _ = expected_calibration_error(p_test, yte)
    coefs = dict(zip(FEATURES, head.coef_[0].tolist()))

    calibration_head_results[model_key] = {
        'auroc': float(auroc), 'brier': float(brier), 'ece': float(ece_h),
        'coefficients': coefs, 'intercept': float(head.intercept_[0]),
        'n_train': len(Xtr), 'n_test': len(Xte),
    }
    print(f'{model_key:<15} AUROC={auroc:.3f}  Brier={brier:.3f}  ECE={ece_h:.3f}')
    print(f'  coefficients: {coefs}')

with open(OUT_DIR / 'data' / 'calibration_head_results.json', 'w') as f:
    json.dump(calibration_head_results, f, indent=2)

## Section 11 — Strategy comparison (Slide 14)

Compare four strategies at a matched abstention rate (~30%):
1. **No calibration**: always answer (baseline)
2. **Entropy threshold**: abstain when avg entropy > τ
3. **Self-confidence threshold**: abstain when self_conf < c
4. **Trained head**: abstain when p_correct < 0.5

In [ ]:
def find_threshold_for_abstention(scores, target_abstention=0.3, abstain_above=True):
    """Pick a threshold so the abstention rate is approximately `target_abstention`."""
    s = np.sort(scores)
    if abstain_above:
        return float(np.quantile(s, 1 - target_abstention))  # abstain on top tail
    else:
        return float(np.quantile(s, target_abstention))      # abstain on bottom tail

TARGET_ABSTENTION = 0.30
strategy_comparison = {}

for model_key, results in all_results.items():
    df = pd.DataFrame(results)
    n_total = len(df)
    y = df['semantic_correct'].astype(int).values

    rows = []

    # 1) No calibration
    rows.append({'strategy': 'No calibration',
                 'accuracy': float(y.mean()),
                 'hallucination_rate': float((y == 0).mean()),
                 'abstention_rate': 0.0,
                 'cond_accuracy': float(y.mean())})

    # 2) Entropy threshold (abstain when entropy too high)
    tau_e = find_threshold_for_abstention(df['avg_entropy'].values, TARGET_ABSTENTION, abstain_above=True)
    ans_e = df['avg_entropy'].values <= tau_e
    rows.append({'strategy': f'Entropy ≤ {tau_e:.2f}',
                 'accuracy': float((ans_e & (y == 1)).mean()),
                 'hallucination_rate': float((ans_e & (y == 0)).mean()),
                 'abstention_rate': float(1 - ans_e.mean()),
                 'cond_accuracy': float(y[ans_e].mean()) if ans_e.any() else 0.0})

    # 3) Self-confidence threshold (abstain when too unconfident)
    c_sc = find_threshold_for_abstention(df['self_confidence'].values, TARGET_ABSTENTION, abstain_above=False)
    ans_s = df['self_confidence'].values >= c_sc
    rows.append({'strategy': f'Self-conf ≥ {c_sc:.2f}',
                 'accuracy': float((ans_s & (y == 1)).mean()),
                 'hallucination_rate': float((ans_s & (y == 0)).mean()),
                 'abstention_rate': float(1 - ans_s.mean()),
                 'cond_accuracy': float(y[ans_s].mean()) if ans_s.any() else 0.0})

    # 4) Trained head (refit on full data here for the comparison; report both)
    if model_key in calibration_head_results:
        X = df[FEATURES].values
        head = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X, y)
        p_corr = head.predict_proba(X)[:, 1]
        c_h = find_threshold_for_abstention(p_corr, TARGET_ABSTENTION, abstain_above=False)
        ans_h = p_corr >= c_h
        rows.append({'strategy': f'Trained head ≥ {c_h:.2f}',
                     'accuracy': float((ans_h & (y == 1)).mean()),
                     'hallucination_rate': float((ans_h & (y == 0)).mean()),
                     'abstention_rate': float(1 - ans_h.mean()),
                     'cond_accuracy': float(y[ans_h].mean()) if ans_h.any() else 0.0})

    strategy_comparison[model_key] = rows

# Print and save
for k, rows in strategy_comparison.items():
    print(f'\n=== {k} (target abstention = {TARGET_ABSTENTION:.0%}) ===')
    print(f'{"Strategy":<25} {"Acc":<8} {"Hall":<8} {"Abst":<8} {"CondAcc":<8}')
    for r in rows:
        print(f'{r["strategy"]:<25} {r["accuracy"]:.3f}   {r["hallucination_rate"]:.3f}   {r["abstention_rate"]:.3f}   {r["cond_accuracy"]:.3f}')

with open(OUT_DIR / 'data' / 'strategy_comparison.json', 'w') as f:
    json.dump(strategy_comparison, f, indent=2)

In [ ]:
# Bar plot: strategy comparison across models
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
model_names = list(strategy_comparison.keys())
strat_labels = ['No calibration', 'Entropy', 'Self-conf', 'Trained head']
x = np.arange(len(model_names))
width = 0.2
colors = ['#888', '#4c72b0', '#dd8452', '#55a868']

for ax, metric, title in [(axes[0], 'cond_accuracy', 'Conditional accuracy on answered (↑ better)'),
                          (axes[1], 'hallucination_rate', 'Hallucination rate (↓ better)')]:
    for s_idx, label in enumerate(strat_labels):
        vals = []
        for mk in model_names:
            row = next((r for r in strategy_comparison[mk] if r['strategy'].startswith(label.split()[0])), None)
            vals.append(row[metric] if row else 0.0)
        ax.bar(x + (s_idx - 1.5)*width, vals, width, label=label, color=colors[s_idx])
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15)
    ax.set_title(title); ax.grid(True, axis='y', alpha=0.3)
    ax.legend(loc='best', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / 'figures' / 'fig_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 12 — Save final summary for the report

This cell aggregates everything into one JSON that the report-generation notebook reads. **Download `final_summary.json` from Drive after this cell runs.**

In [ ]:
summary = {
    'config': {
        'models': MODELS, 'n_questions': N_QUESTIONS, 'max_new_tokens': MAX_NEW_TOKENS,
        'entropy_thresholds': ENTROPY_THRESHOLDS,
        'sem_sim_threshold': SEM_SIM_THRESHOLD,
        'target_abstention': TARGET_ABSTENTION,
        'seed': SEED,
    },
    'baseline_metrics': baseline,
    'abstention_curves': abstention_curves,
    'ece_results': {k: {kk: vv for kk, vv in v.items() if kk in ('mc1_ece','self_conf_ece')}
                    for k, v in ece_results.items()},
    'calibration_head_results': calibration_head_results,
    'strategy_comparison': strategy_comparison,
}
with open(OUT_DIR / 'final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved consolidated summary to {OUT_DIR / "final_summary.json"}')
print('Figures available in', OUT_DIR / 'figures')
print('Per-question raw results available in', OUT_DIR / 'data')

## Section 13 — Qualitative inspection (for the report)

Show 3 examples per model: a confident-correct, a confident-wrong (hallucination), and an unconfident-wrong. These quotes/snippets are useful illustrations for the discussion section.

In [ ]:
qualitative = {}
for model_key, results in all_results.items():
    df = pd.DataFrame(results).copy()
    df['confidence'] = np.exp(df['avg_logprob'])
    examples = {}
    correct  = df[df['semantic_correct'] == 1].sort_values('confidence', ascending=False)
    wrong    = df[df['semantic_correct'] == 0]
    if len(correct) > 0:
        examples['confident_correct']    = correct.iloc[0][['question','generated','best_correct','confidence','avg_entropy','self_confidence']].to_dict()
    if len(wrong) > 0:
        confident_wrong = wrong.sort_values('confidence', ascending=False)
        unconfident_wrong = wrong.sort_values('confidence', ascending=True)
        examples['confident_wrong']      = confident_wrong.iloc[0][['question','generated','best_correct','confidence','avg_entropy','self_confidence']].to_dict()
        examples['unconfident_wrong']    = unconfident_wrong.iloc[0][['question','generated','best_correct','confidence','avg_entropy','self_confidence']].to_dict()
    qualitative[model_key] = examples

with open(OUT_DIR / 'data' / 'qualitative_examples.json', 'w') as f:
    json.dump(qualitative, f, indent=2, default=str)

for k, ex in qualitative.items():
    print(f'\n=== {k} ===')
    for cat, e in ex.items():
        print(f'  [{cat}]')
        print(f'    Q: {e["question"]}')
        print(f'    Generated: {e["generated"]}')
        print(f'    Correct:   {e["best_correct"]}')
        print(f'    conf={e["confidence"]:.3f}  H={e["avg_entropy"]:.2f}  self_conf={e["self_confidence"]:.2f}')

## Done!

**Files produced:**
- `final_summary.json` — top-level metrics for the report
- `data/{model}_results.jsonl` — per-question raw results
- `data/baseline_metrics.json`, `abstention_curves.json`, `ece_results.json`, `calibration_head_results.json`, `strategy_comparison.json`, `qualitative_examples.json`
- `figures/fig_uncertainty_vs_correctness.png`, `fig_abstention_sweep.png`, `fig_reliability.png`, `fig_strategy_comparison.png`

Next: download these files locally, then run the **report builder** notebook (separate file) to generate the .docx final report.